In [11]:
"""
Flight Delay Prediction — Streamlit App
-----------------------------------------
Supports two loading modes:
  1. PORTABLE  → model_export/ folder (Fix 3, works across sklearn versions)
  2. JOBLIB    → flight_delay_model.pkl (Fix 2, requires matching sklearn version)

Run:  streamlit run app.py
"""

import os, json
import numpy as np
import pandas as pd
import streamlit as st

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Flight Delay Predictor",
    page_icon="✈️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────
# LOAD ARTIFACT  — tries portable first, then pkl
# ─────────────────────────────────────────────
@st.cache_resource(show_spinner="Loading model…")
def load_artifact():
    """
    Priority:
      1. model_export/metadata.json  — portable (Fix 3)
      2. flight_delay_model.pkl      — joblib   (Fix 2)
    """
    # ── Mode 1: portable ──────────────────────────────────────────
    if os.path.exists("model_export/metadata.json"):
        with open("model_export/metadata.json") as f:
            meta = json.load(f)
        with open("model_export/preprocessor_state.json") as f:
            ps = json.load(f)

        native_fmt = meta.get("native_format", "")
        model_obj  = None

        if native_fmt == "xgboost_json":
            try:
                import xgboost as xgb
                model_obj = xgb.XGBClassifier()
                model_obj.load_model("model_export/model.json")
            except Exception as e:
                st.error(f"Failed to load XGBoost model: {e}")
                st.stop()

        elif native_fmt == "sklearn_rf_pkl":
            try:
                import joblib
                model_obj = joblib.load("model_export/model_rf.pkl")
            except Exception as e:
                st.error(f"Failed to load Random Forest: {e}")
                st.stop()

        elif native_fmt == "logistic_numpy":
            try:
                from sklearn.linear_model import LogisticRegression
                coef      = np.load("model_export/lr_coef.npy")
                intercept = np.load("model_export/lr_intercept.npy")
                model_obj = LogisticRegression()
                model_obj.coef_      = coef
                model_obj.intercept_ = intercept
                model_obj.classes_   = np.array([0, 1])
            except Exception as e:
                st.error(f"Failed to load Logistic Regression: {e}")
                st.stop()

        return {"mode": "portable", "meta": meta, "ps": ps, "model": model_obj}

    # ── Mode 2: joblib pkl ────────────────────────────────────────
    if os.path.exists("flight_delay_model.pkl"):
        try:
            import joblib, sklearn
            artifact = joblib.load("flight_delay_model.pkl")
            saved_ver = artifact.get("_versions", {}).get("sklearn", "unknown")
            curr_ver  = sklearn.__version__
            if saved_ver != curr_ver and saved_ver != "unknown":
                st.warning(
                    f"⚠️ **sklearn version mismatch**: model saved with `{saved_ver}`, "
                    f"running `{curr_ver}`.  \n"
                    f"Run **Fix 3** in the notebook to generate a portable export, or:  \n"
                    f"```bash\npip install scikit-learn=={saved_ver} --force-reinstall\n```"
                )
            return {"mode": "joblib", "artifact": artifact}
        except AttributeError as e:
            if "_RemainderColsList" in str(e) or "Can't get attribute" in str(e):
                st.error(
                    "### ❌ sklearn Version Mismatch\n\n"
                    f"`{e}`\n\n"
                    "**Quick fix — in your terminal:**\n"
                    "```bash\nconda activate ml\n"
                    "pip install scikit-learn==1.3.2 --force-reinstall\n```\n\n"
                    "**Best fix — run Section Fix 3 in the notebook** to generate a "
                    "portable `model_export/` folder that works with any sklearn version."
                )
                st.stop()
            raise

    st.error(
        "**No model found.**  \n"
        "Expected either `model_export/` folder (portable) or "
        "`flight_delay_model.pkl` (joblib) in the same directory as `app.py`."
    )
    st.stop()


loaded = load_artifact()

# ─────────────────────────────────────────────
# UNPACK  — normalise both modes to same interface
# ─────────────────────────────────────────────
if loaded["mode"] == "portable":
    meta              = loaded["meta"]
    ps                = loaded["ps"]
    _model            = loaded["model"]
    NUMERIC_FEATURES  = ps["numeric_features"]
    CATEGORICAL_FEATURES = ps["categorical_features"]
    model_name        = meta["model_name"]
    metrics           = meta["metrics"]
    airline_codes     = meta["airline_codes"]
    origin_codes      = meta["origin_codes"]
    dest_codes        = meta["dest_codes"]
    dep_time_blocks   = meta["dep_time_blocks"]
    airport_lookup    = meta["airport_lookup"]
    airline_lookup    = meta["airline_lookup"]
    airline_hist_map  = meta["airline_hist_map"]
    origin_hist_map   = meta["origin_hist_map"]
    avg_taxiout_map   = meta["avg_taxiout_map"]
    avg_daily_flights_map = meta["avg_daily_flights_map"]
    global_delay_rate = meta["global_delay_rate"]
    global_taxiout    = meta["global_taxiout"]
    global_daily_flights = meta["global_daily_flights"]
    train_rows        = meta["train_rows"]
    test_rows         = meta["test_rows"]

else:  # joblib
    art               = loaded["artifact"]
    _model            = art["pipeline"]       # full sklearn pipeline
    NUMERIC_FEATURES  = art["numeric_features"]
    CATEGORICAL_FEATURES = art["categorical_features"]
    model_name        = art["model_name"]
    metrics           = art["metrics"]
    airline_codes     = art["airline_codes"]
    origin_codes      = art["origin_codes"]
    dest_codes        = art["dest_codes"]
    dep_time_blocks   = art["dep_time_blocks"]
    airport_lookup    = art["airport_lookup"]
    airline_lookup    = art["airline_lookup"]
    airline_hist_map  = art["airline_hist_map"]
    origin_hist_map   = art["origin_hist_map"]
    avg_taxiout_map   = art["avg_taxiout_map"]
    avg_daily_flights_map = art["avg_daily_flights_map"]
    global_delay_rate = art["global_delay_rate"]
    global_taxiout    = art["global_taxiout"]
    global_daily_flights = art["global_daily_flights"]
    train_rows        = art["train_rows"]
    test_rows         = art["test_rows"]
    ps                = None

# ─────────────────────────────────────────────
# PORTABLE TRANSFORM  (used only in portable mode)
# ─────────────────────────────────────────────
def _portable_transform(X_raw: pd.DataFrame) -> np.ndarray:
    """Reproduce ColumnTransformer manually using saved state."""
    num_cols = ps["numeric_features"]
    cat_cols = ps["categorical_features"]

    # Numeric — impute then standardise
    X_num  = X_raw[num_cols].copy().astype(float)
    fill   = np.array(ps["num_imputer_statistics"])
    for i, col in enumerate(num_cols):
        X_num[col] = X_num[col].fillna(fill[i])
    X_num_scaled = (X_num.values - np.array(ps["scaler_mean"])) / np.array(ps["scaler_scale"])

    # Categorical — impute then one-hot
    X_cat  = X_raw[cat_cols].copy().astype(str)
    cat_fill = ps["cat_imputer_statistics"]
    for i, col in enumerate(cat_cols):
        X_cat[col] = X_cat[col].replace("nan", cat_fill[i]).fillna(cat_fill[i])

    ohe_blocks = []
    for i, col in enumerate(cat_cols):
        cats  = ps["ohe_categories"][i]
        vals  = X_cat[col].values
        block = np.zeros((len(vals), len(cats)), dtype=np.float32)
        for j, cat in enumerate(cats):
            block[:, j] = (vals == str(cat)).astype(float)
        ohe_blocks.append(block)

    return np.hstack([X_num_scaled, np.hstack(ohe_blocks)])


def _predict(X_raw: pd.DataFrame):
    """Returns (label, proba) — works in both portable and joblib modes."""
    if loaded["mode"] == "portable":
        X_t   = _portable_transform(X_raw)
        proba = float(_model.predict_proba(X_t)[0, 1])
        label = int(proba >= 0.5)
    else:
        label = int(_model.predict(X_raw)[0])
        proba = float(_model.predict_proba(X_raw)[0, 1])
    return label, proba


def _predict_batch(X_raw: pd.DataFrame):
    """Returns (labels array, probas array) for batch scoring."""
    if loaded["mode"] == "portable":
        X_t    = _portable_transform(X_raw)
        probas = _model.predict_proba(X_t)[:, 1]
        labels = (probas >= 0.5).astype(int)
    else:
        labels = _model.predict(X_raw)
        probas = _model.predict_proba(X_raw)[:, 1]
    return labels, probas

# ─────────────────────────────────────────────
# CONSTANTS & HELPERS
# ─────────────────────────────────────────────
MONTH_NAMES = {
    1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
    7:"July",8:"August",9:"September",10:"October",11:"November",12:"December",
}
DAY_NAMES = {
    0:"Monday",1:"Tuesday",2:"Wednesday",3:"Thursday",
    4:"Friday",5:"Saturday",6:"Sunday",
}
AIRLINE_FULL = {
    "AA":"American Airlines","AS":"Alaska Airlines","B6":"JetBlue Airways",
    "DL":"Delta Air Lines","F9":"Frontier Airlines","G4":"Allegiant Air",
    "HA":"Hawaiian Airlines","NK":"Spirit Airlines","UA":"United Airlines",
    "WN":"Southwest Airlines",
}

def fmt_airport(code):
    city = airport_lookup.get(code, "")
    return f"{code} — {city}" if city else code

def fmt_airline(code):
    name = AIRLINE_FULL.get(code, airline_lookup.get(code, ""))
    return f"{code} — {name}" if name else code

def get_season(month):
    return {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
            6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"}[month]

def is_holiday_window(month, day):
    return int(
        (month==12 and day>=15) or (month==1 and day<=7) or
        (month==6 and day>=15) or (month==7 and day<=7) or
        (month==11 and day>=20)
    )

def build_feature_row(airline, origin, dest,
                      dep_hh, dep_mm, arr_hh, arr_mm,
                      elapsed_min, distance, dep_time_blk,
                      month, day_of_month, day_of_week):
    dep_time_int = dep_hh * 100 + dep_mm
    arr_time_int = arr_hh * 100 + arr_mm
    dep_hour     = dep_hh

    is_weekend   = int(day_of_week in (5, 6))
    is_peak_hour = int(dep_hour in list(range(7,10)) + list(range(16,20)))
    season       = get_season(month)
    holiday      = is_holiday_window(month, day_of_month)

    airline_hist  = airline_hist_map.get(airline, global_delay_rate)
    origin_hist   = origin_hist_map.get(origin,   global_delay_rate)
    avg_to        = avg_taxiout_map.get(origin,    global_taxiout)
    daily_flights = avg_daily_flights_map.get(origin, global_daily_flights)

    dist_bucket = pd.cut([distance],
                         bins=[0,300,600,1000,1500,9999],
                         labels=["Short","Medium","Long","X-Long","Ultra"])[0]

    row = {
        "CRSDepTime": dep_time_int, "CRSArrTime": arr_time_int,
        "CRSElapsedTime": elapsed_min, "Distance": distance,
        "Month": month, "DayofMonth": day_of_month, "DayOfWeek": day_of_week,
        "DepHour": dep_hour, "IsWeekend": is_weekend, "IsPeakHour": is_peak_hour,
        "HolidayWindow": holiday, "OriginDailyFlights": daily_flights,
        "AirlineHistDelayRate": airline_hist, "OriginHistDelayRate": origin_hist,
        "AvgOriginTaxiOut": avg_to,
        "DepHour_sin":    np.sin(2*np.pi*dep_hour/24),
        "DepHour_cos":    np.cos(2*np.pi*dep_hour/24),
        "Month_sin":      np.sin(2*np.pi*month/12),
        "Month_cos":      np.cos(2*np.pi*month/12),
        "DayOfWeek_sin":  np.sin(2*np.pi*day_of_week/7),
        "DayOfWeek_cos":  np.cos(2*np.pi*day_of_week/7),
        "Marketing_Airline_Network": airline,
        "Origin": origin, "Dest": dest,
        "DistanceBucket": str(dist_bucket),
        "Season": season, "DepTimeBlk": dep_time_blk,
    }

    all_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    return pd.DataFrame([{k: row.get(k, np.nan) for k in all_cols}])


def risk_colour(p):
    if p < 0.30: return "#28a745", "LOW RISK"
    if p < 0.55: return "#ffc107", "MODERATE RISK"
    return "#dc3545", "HIGH RISK"

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.title("Model Info")
    mode_badge = "🟢 Portable" if loaded["mode"] == "portable" else "🔵 Joblib"
    st.markdown(f"**Load mode:** {mode_badge}")
    st.markdown(f"**Algorithm:** {model_name}")
    st.markdown(f"**Training rows:** {train_rows:,}")
    st.markdown(f"**Test rows:** {test_rows:,}")

    st.markdown("---")
    st.markdown("### Test-Set Performance")
    if model_name in metrics:
        m = metrics[model_name]
        c1, c2 = st.columns(2)
        c1.metric("ROC-AUC",  f"{m['ROC-AUC']:.3f}")
        c2.metric("F1 Score",  f"{m['F1']:.3f}")
        c1.metric("Recall",    f"{m['Recall']:.3f}")
        c2.metric("Precision", f"{m['Precision']:.3f}")

    st.markdown("---")
    st.markdown("### All Models")
    rows = [{"Model": n, "AUC": round(m["ROC-AUC"],3),
             "F1": round(m["F1"],3), "Recall": round(m["Recall"],3)}
            for n, m in metrics.items()]
    st.dataframe(pd.DataFrame(rows).set_index("Model"), use_container_width=True)

    st.markdown("---")
    st.caption("Target: ArrDel15=1 when arrival delay > 15 min. "
               "Only scheduling-time features — no post-departure leakage.")

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
st.title("✈️ Flight Delay Predictor")
st.markdown(
    "Enter flight scheduling details below. The model predicts whether your flight "
    "will arrive **more than 15 minutes late**."
)
st.markdown("---")

with st.form("prediction_form"):
    st.subheader("Flight Details")
    col1, col2, col3 = st.columns(3)
    with col1:
        airline_sel = st.selectbox("Airline", airline_codes, format_func=fmt_airline,
                                   index=airline_codes.index("DL") if "DL" in airline_codes else 0)
    with col2:
        origin_sel  = st.selectbox("Origin Airport", origin_codes, format_func=fmt_airport,
                                   index=origin_codes.index("ATL") if "ATL" in origin_codes else 0)
    with col3:
        dest_sel    = st.selectbox("Destination Airport", dest_codes, format_func=fmt_airport,
                                   index=dest_codes.index("ORD") if "ORD" in dest_codes else 0)

    st.markdown("#### Schedule")
    col4, col5, col6 = st.columns(3)
    with col4:
        dep_hh = st.slider("Departure Hour (24h)", 0, 23, 9)
        dep_mm = st.selectbox("Departure Minute", [0, 15, 30, 45], index=0)
    with col5:
        arr_hh = st.slider("Arrival Hour (24h)", 0, 23, 11)
        arr_mm = st.selectbox("Arrival Minute", [0, 15, 30, 45], index=0)
    with col6:
        elapsed_min = st.number_input("Scheduled Duration (min)", 20, 700, 120, step=5)

    st.markdown("#### Route Info")
    col7, col8 = st.columns(2)
    with col7:
        distance = st.number_input("Distance (miles)", 30, 5000, 800, step=10)
        dep_blk  = st.selectbox("Departure Time Block",
                                 dep_time_blocks if dep_time_blocks else ["0900-0959"],
                                 index=2 if len(dep_time_blocks) > 2 else 0)
    with col8:
        month       = st.selectbox("Month", list(MONTH_NAMES.keys()),
                                   format_func=lambda m: MONTH_NAMES[m], index=0)
        day_of_month = st.slider("Day of Month", 1, 31, 15)
        day_of_week  = st.selectbox("Day of Week", list(DAY_NAMES.keys()),
                                    format_func=lambda d: DAY_NAMES[d], index=0)

    submitted = st.form_submit_button("🔍 Predict Delay", use_container_width=True)

# ─────────────────────────────────────────────
# RESULT
# ─────────────────────────────────────────────
if submitted:
    X_input    = build_feature_row(airline_sel, origin_sel, dest_sel,
                                   dep_hh, dep_mm, arr_hh, arr_mm,
                                   elapsed_min, distance, dep_blk,
                                   month, day_of_month, day_of_week)
    prediction, proba = _predict(X_input)
    colour, risk_label = risk_colour(proba)

    st.markdown("---")
    st.subheader("Prediction Result")

    r1, r2, r3 = st.columns([2, 2, 3])
    with r1:
        if prediction == 1:
            st.error("### 🔴 DELAYED\nFlight likely to arrive **>15 min late**")
        else:
            st.success("### 🟢 ON TIME\nFlight likely to arrive **within 15 min**")

    with r2:
        st.markdown(f"""
        <div style='border-left:6px solid {colour}; padding:12px 20px;
                    border-radius:8px; background:#1e1e2e;'>
            <div style='font-size:2.2rem; font-weight:800; color:{colour};'>{proba*100:.1f}%</div>
            <div style='color:#ccc; font-size:0.95rem;'>Delay Probability</div>
            <div style='font-size:1.1rem; font-weight:700; color:{colour}; margin-top:6px;'>{risk_label}</div>
        </div>""", unsafe_allow_html=True)

    with r3:
        st.markdown("**Delay Probability Gauge**")
        st.progress(proba)
        st.markdown(
            "<div style='display:flex; justify-content:space-between; font-size:0.8rem;'>"
            "<span style='color:#28a745'>0% — Very Safe</span>"
            "<span style='color:#dc3545'>100% — Very Likely</span>"
            "</div>", unsafe_allow_html=True)

    # ── Profile summary ──
    st.markdown("---")
    st.subheader("Flight Profile Summary")
    season     = get_season(month)
    is_holiday = bool(is_holiday_window(month, day_of_month))
    is_peak    = dep_hh in list(range(7,10)) + list(range(16,20))
    is_wknd    = day_of_week in (5, 6)

    tags = []
    if is_holiday: tags.append("🎄 Holiday Window")
    if is_peak:    tags.append("⚡ Peak Hour")
    if is_wknd:    tags.append("📅 Weekend")
    tags.append(f"🌤 {season}")

    tag_html = " &nbsp; ".join(
        f"<span style='background:#333;padding:4px 10px;border-radius:12px;font-size:0.85rem;'>{t}</span>"
        for t in tags)
    st.markdown(tag_html, unsafe_allow_html=True)
    st.markdown("")

    i1, i2, i3, i4 = st.columns(4)
    i1.metric("Route",    f"{origin_sel} → {dest_sel}")
    i2.metric("Airline",  fmt_airline(airline_sel))
    i3.metric("Distance", f"{distance:,} mi")
    i4.metric("Duration", f"{elapsed_min} min")

    # ── Risk factors ──
    st.markdown("---")
    st.subheader("Risk Factor Analysis")
    a_rate = airline_hist_map.get(airline_sel, global_delay_rate)
    o_rate = origin_hist_map.get(origin_sel,   global_delay_rate)
    avg_to = avg_taxiout_map.get(origin_sel,   global_taxiout)

    def _impact(val, hi=0.30, lo=0.20):
        return "🔴 High" if val>hi else "🟡 Medium" if val>lo else "🟢 Low"

    factor_data = {
        "Risk Factor": ["Airline Delay History","Airport Delay History",
                        "Departure Hour","Holiday / Season","Avg Taxi-Out"],
        "Value":       [f"{a_rate*100:.1f}%", f"{o_rate*100:.1f}%",
                        f"Hour {dep_hh:02d}:00 ({'Peak' if is_peak else 'Off-peak'})",
                        f"{'YES' if is_holiday else 'No'} / {season}",
                        f"{avg_to:.1f} min"],
        "Impact":      [_impact(a_rate), _impact(o_rate),
                        "🔴 High" if is_peak else "🟢 Low",
                        "🔴 High" if is_holiday else ("🟡 Medium" if season in ("Summer","Winter") else "🟢 Low"),
                        "🔴 High" if avg_to>20 else ("🟡 Medium" if avg_to>15 else "🟢 Low")],
    }
    st.dataframe(pd.DataFrame(factor_data), use_container_width=True, hide_index=True)

    # ── Advice ──
    st.markdown("---")
    if proba >= 0.55:
        st.warning("**High delay risk.** Consider an earlier departure, a more reliable "
                   "airline, or extra connection time.")
    elif proba >= 0.30:
        st.info("**Moderate delay risk.** Monitor flight status on the day of travel.")
    else:
        st.success("**Low delay risk.** Early-morning flights from reliable airlines at "
                   "low-congestion airports have the best on-time performance.")

# ─────────────────────────────────────────────
# BATCH PREDICTION
# ─────────────────────────────────────────────
st.markdown("---")
st.subheader("Batch Prediction (CSV Upload)")
st.markdown("Upload a flight CSV — the app will score every row and let you download the results.")

uploaded = st.file_uploader("Upload flight CSV", type=["csv"])
if uploaded is not None:
    try:
        batch_df = pd.read_csv(uploaded)
        st.write(f"Loaded {len(batch_df):,} rows × {batch_df.shape[1]} columns.")

        # Reproduce feature engineering
        batch_df["FlightDate"] = pd.to_datetime(batch_df["FlightDate"], errors="coerce")
        batch_df["Year"]       = batch_df["FlightDate"].dt.year
        batch_df["Month"]      = batch_df["FlightDate"].dt.month
        batch_df["DayofMonth"] = batch_df["FlightDate"].dt.day
        batch_df["DayOfWeek"]  = batch_df["FlightDate"].dt.dayofweek
        batch_df["DepHour"]    = (batch_df["CRSDepTime"] // 100).clip(0, 23)
        batch_df["IsWeekend"]  = batch_df["DayOfWeek"].isin([5,6]).astype(int)
        batch_df["IsPeakHour"] = batch_df["DepHour"].isin(list(range(7,10))+list(range(16,20))).astype(int)
        batch_df["Season"]     = batch_df["Month"].map(
            {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
             6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"})
        batch_df["HolidayWindow"] = (
            ((batch_df["Month"]==12)&(batch_df["DayofMonth"]>=15)) |
            ((batch_df["Month"]== 1)&(batch_df["DayofMonth"]<= 7)) |
            ((batch_df["Month"]== 6)&(batch_df["DayofMonth"]>=15)) |
            ((batch_df["Month"]== 7)&(batch_df["DayofMonth"]<= 7)) |
            ((batch_df["Month"]==11)&(batch_df["DayofMonth"]>=20))
        ).astype(int)
        batch_df["AirlineHistDelayRate"]  = batch_df["Marketing_Airline_Network"].map(airline_hist_map).fillna(global_delay_rate)
        batch_df["OriginHistDelayRate"]   = batch_df["Origin"].map(origin_hist_map).fillna(global_delay_rate)
        batch_df["AvgOriginTaxiOut"]      = batch_df["Origin"].map(avg_taxiout_map).fillna(global_taxiout)
        batch_df["OriginDailyFlights"]    = batch_df["Origin"].map(avg_daily_flights_map).fillna(global_daily_flights)
        for col, period in [("DepHour",24),("Month",12),("DayOfWeek",7)]:
            batch_df[f"{col}_sin"] = np.sin(2*np.pi*batch_df[col]/period)
            batch_df[f"{col}_cos"] = np.cos(2*np.pi*batch_df[col]/period)
        batch_df["DistanceBucket"] = pd.cut(batch_df["Distance"],
            bins=[0,300,600,1000,1500,9999],
            labels=["Short","Medium","Long","X-Long","Ultra"]).astype(str)

        X_batch = batch_df[[c for c in NUMERIC_FEATURES+CATEGORICAL_FEATURES if c in batch_df.columns]]
        preds, probas = _predict_batch(X_batch)

        batch_df["PredictedDelay"]   = preds
        batch_df["DelayProbability"] = probas.round(3)
        batch_df["DelayLabel"]       = batch_df["PredictedDelay"].map({0:"On-Time",1:"Delayed"})

        st.success(f"Scored {len(batch_df):,} flights.")
        display_cols = [c for c in ["FlightDate","Marketing_Airline_Network","Origin","Dest",
                                     "CRSDepTime","DelayLabel","DelayProbability"] if c in batch_df.columns]
        st.dataframe(batch_df[display_cols].head(50), use_container_width=True)

        st.download_button(
            "⬇️ Download Predictions CSV",
            data=batch_df.to_csv(index=False).encode("utf-8"),
            file_name="flight_delay_predictions.csv",
            mime="text/csv",
            use_container_width=True,
        )
    except Exception as e:
        st.error(f"Error processing file: {e}")

2026-06-11 11:48:19.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 11:48:19.022 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-06-11 11:48:19.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 11:48:19.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 11:48:19.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 11:48:19.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


AttributeError: Can't get attribute '_RemainderColsList' on <module 'sklearn.compose._column_transformer' from '/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py'>